# ==============================================================================
# PREREQUISITE LIBRARIES FOR THIS NOTEBOOK
# ==============================================================================
# To run this pipeline, install the required dependencies using pip:
#
#   pip install torch numpy scipy cleanlab scikit-learn
#
# Package Breakdown:
#   - torch        : Deep learning framework for loss correction modules
#   - numpy        : Matrix calculations & transition matrix ops
#   - scipy        : Hypothesis testing (z-test, p-values, CI bounds)
#   - cleanlab     : Confident learning algorithms for label noise detection
#   - scikit-learn : Baseline classification & evaluation metrics
# ==============================================================================

In [ ]:
pip install torch numpy scipy cleanlab scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.1/306.1 kB 8.0 MB/s eta 0:00:00


In [ ]:
import math
import numpy as np
import scipy.stats as stats
import torch
import torch.nn as nn
import torch.nn.functional as F
from cleanlab.filter import find_label_issues

# ==========================================
# 1. MILESTONE 1: STATISTICAL NOISE ANALYSIS
# ==========================================

class LabelNoiseAnalyzer:
    """Performs statistical audits and hypothesis testing on label noise."""

    @staticmethod
    def calculate_confidence_interval(n_audited: int = 400, n_disagreements: int = 60, confidence: float = 0.95):
        p_hat = n_disagreements / n_audited
        z_score = stats.norm.ppf(1 - (1 - confidence) / 2)
        se = math.sqrt(p_hat * (1 - p_hat) / n_audited)
        margin_error = z_score * se
        ci_lower, ci_upper = p_hat - margin_error, p_hat + margin_error

        print(f"--- Milestone 1: Noise Rate Audit ---")
        print(f"Sample Noise Rate (p_hat): {p_hat:.2%}")
        print(f"Standard Error (SE): {se:.4f}")
        print(f"95% Confidence Interval: [{ci_lower:.2%}, {ci_upper:.2%}]\n")
        return p_hat, (ci_lower, ci_upper)

    @staticmethod
    def run_two_proportion_z_test(nA: int = 180, xA: int = 12, nB: int = 220, xB: int = 48):
        pA = xA / nA
        pB = xB / nB
        p_pooled = (xA + xB) / (nA + nB)

        se_diff = math.sqrt(p_pooled * (1 - p_pooled) * ((1 / nA) + (1 / nB)))
        z_stat = (pB - pA) / se_diff
        p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

        print(f"--- Milestone 1: Tier A vs Tier B Comparison ---")
        print(f"Tier A Noise Rate: {pA:.2%}")
        print(f"Tier B Noise Rate: {pB:.2%}")
        print(f"Calculated z-statistic: {z_stat:.4f}")
        print(f"p-value: {p_value:.6e} (Statistically Significant: {p_value < 0.05})\n")
        return z_stat, p_value


# ==========================================
# 2. MILESTONE 3: UNIFORM NOISE ACCURACY RECOVERY
# ==========================================

def recover_true_accuracy(observed_accuracy: float = 0.82, noise_rate: float = 0.15, num_classes: int = 6):
    """Solves the inversion formula for uniform symmetric label noise."""
    true_accuracy = (observed_accuracy - (noise_rate / (num_classes - 1))) / (1 - (num_classes / (num_classes - 1)) * noise_rate)
    print(f"--- Milestone 3: Accuracy Recovery ---")
    print(f"Observed Test Accuracy: {observed_accuracy:.2%}")
    print(f"Estimated True Accuracy (under uniform noise): {true_accuracy:.2%}\n")
    return true_accuracy


# ==========================================
# 3. MILESTONE 5: NOISE-ROBUST PYTORCH LOSS MODULE
# ==========================================

class ForwardLossCorrection(nn.Module):
    """Loss module adjusting prediction probabilities via Noise Transition Matrix T."""

    def __init__(self, transition_matrix: torch.Tensor):
        super(ForwardLossCorrection, self).__init__()
        self.T = transition_matrix  # Shape: (K, K)

    def forward(self, logits: torch.Tensor, noisy_targets: torch.Tensor) -> torch.Tensor:
        # Step 1: Compute clean softmax probabilities P(Y | x)
        probs = F.softmax(logits, dim=1)

        # Step 2: Map clean probabilities to noisy domain P(Y_tilde | x) = P(Y | x) @ T
        noisy_probs = torch.matmul(probs, self.T)

        # Step 3: Compute Cross-Entropy on adjusted distribution
        log_noisy_probs = torch.log(torch.clamp(noisy_probs, min=1e-12))
        return F.nll_loss(log_noisy_probs, noisy_targets)


# ==========================================
# 4. MILESTONE 5: CONFIDENT LEARNING CLEANLAB PIPELINE
# ==========================================

def run_confident_learning_pruning(predicted_probabilities: np.ndarray, noisy_labels: np.ndarray):
    """Uses Cleanlab to identify corrupt labels for targeted human review."""
    print(f"--- Milestone 4 & 5: Confident Learning (Cleanlab) ---")

    # Identify indices of probable label issues
    label_issues_mask = find_label_issues(
        labels=noisy_labels,
        pred_probs=predicted_probabilities,
        return_indices_ranked_by="self_confidence"
    )

    print(f"Total noisy instances identified by Cleanlab: {len(label_issues_mask)}")
    return label_issues_mask


# ==========================================
# MAIN EXECUTION ENTRY POINT
# ==========================================

if __name__ == "__main__":
    print("==================================================")
    print(" TriNex-AI: HelpDeskNet-50K Solution Execution")
    print("==================================================\n")

    # 1. Run Statistical Audit calculations
    analyzer = LabelNoiseAnalyzer()
    analyzer.calculate_confidence_interval(n_audited=400, n_disagreements=60)
    analyzer.run_two_proportion_z_test(nA=180, xA=12, nB=220, xB=48)

    # 2. Estimate True Model Accuracy
    recover_true_accuracy(observed_accuracy=0.82, noise_rate=0.15, num_classes=6)

    # 3. Test Forward Loss Correction PyTorch Module
    # Example transition matrix T for 6 classes
    T_matrix = torch.eye(6) * 0.85 + torch.ones(6, 6) * 0.03
    T_matrix = T_matrix / T_matrix.sum(dim=1, keepdim=True)  # Normalize rows

    loss_fn = ForwardLossCorrection(transition_matrix=T_matrix)
    dummy_logits = torch.randn(8, 6)  # Batch of 8 samples, 6 classes
    dummy_targets = torch.tensor([0, 1, 2, 3, 4, 5, 0, 1])

    calculated_loss = loss_fn(dummy_logits, dummy_targets)
    print(f"--- PyTorch Test ---")
    print(f"Forward-Corrected Loss output: {calculated_loss.item():.4f}\n")
    print("Pipeline execution complete! All modules operational.")

 TriNex-AI: HelpDeskNet-50K Solution Execution

--- Milestone 1: Noise Rate Audit ---
Sample Noise Rate (p_hat): 15.00%
Standard Error (SE): 0.0179
95% Confidence Interval: [11.50%, 18.50%]

--- Milestone 1: Tier A vs Tier B Comparison ---
Tier A Noise Rate: 6.67%
Tier B Noise Rate: 21.82%
Calculated z-statistic: 4.2220
p-value: 2.421406e-05 (Statistically Significant: True)

--- Milestone 3: Accuracy Recovery ---
Observed Test Accuracy: 82.00%
Estimated True Accuracy (under uniform noise): 96.34%

--- PyTorch Test ---
Forward-Corrected Loss output: 1.6498

Pipeline execution complete! All modules operational.
